In [21]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [22]:
spark = (
    SparkSession.builder
    .appName("aula_06_agregacoes")
    .getOrCreate()
)

In [23]:
df = spark.read.csv("/workspace/datasets/06_agregacoes/vendas.csv", header= True, inferSchema= True)
df.show()

+--------+----------+-----------+--------------------+--------------------+----------+--------------+-----------+--------------------+------+---------------+
|venda_id|data_venda|vendedor_id|             produto|           categoria|quantidade|valor_unitario|valor_total|              cidade|estado|forma_pagamento|
+--------+----------+-----------+--------------------+--------------------+----------+--------------+-----------+--------------------+------+---------------+
|       1|2025-01-01|        106|   Kit Halteres 20kg|     Esporte e Lazer|         3|         259.9|      779.7|              Recife|    PE| Cartão Crédito|
|       2|2025-01-01|        106|Carregador Turbo ...|Celulares e Acess...|         5|         80.91|     404.55|           São Paulo|    SP|         Boleto|
|       3|2025-01-01|        110|Console de Videogame|               Games|         1|        2999.0|     2999.0|            Salvador|    BA|  Cartão Débito|
|       4|2025-01-01|        110|    Notebook Core i

In [24]:
df_agregado = (
    df.groupBy("categoria")
    .agg(F.sum("valor_total").alias("faturamento_total"))
)

In [25]:
df_agregado.show()

+--------------------+------------------+
|           categoria| faturamento_total|
+--------------------+------------------+
|              Móveis| 4068842.090000002|
|     Esporte e Lazer|3568416.5300000054|
|               Games| 3341485.670000002|
|         Informática|  3570334.61999996|
|Celulares e Acess...| 2450861.539999995|
|    Eletrodomésticos| 5686772.190000008|
+--------------------+------------------+



In [26]:
df_agregado.explain(True)

== Parsed Logical Plan ==
'Aggregate ['categoria], ['categoria, sum('valor_total) AS faturamento_total#493]
+- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_pagamento#412] csv

== Analyzed Logical Plan ==
categoria: string, faturamento_total: double
Aggregate [categoria#406], [categoria#406, sum(valor_total#409) AS faturamento_total#493]
+- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_pagamento#412] csv

== Optimized Logical Plan ==
Aggregate [categoria#406], [categoria#406, sum(valor_total#409) AS faturamento_total#493]
+- Project [categoria#406, valor_total#409]
   +- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_pagamento#412] csv

== Physical Plan ==


In [27]:
df_agregado2 = (
    df.groupBy("estado")
    .agg(
        F.sum("valor_total").alias("faturamento_total"),
        F.avg("valor_total").alias("ticket_medio"),
        F.count("*").alias("quantidade_vendas"),
        F.max("valor_total").alias("maior_venda"),
        F.min("valor_total").alias("menor_venda")
    )
)

In [28]:
df_agregado2.show()

+------+------------------+------------------+-----------------+-----------+-----------+
|estado| faturamento_total|      ticket_medio|quantidade_vendas|maior_venda|menor_venda|
+------+------------------+------------------+-----------------+-----------+-----------+
|    GO|1173497.0300000012| 2230.982946768063|              526|     6998.0|       79.9|
|    SP| 4492084.900000003|2257.3290954773884|             1990|     6998.0|      67.92|
|    RS|1238966.5800000012|2290.1415526802243|              541|     6998.0|       75.9|
|    MG|2772609.1599999964|2295.2062582781427|             1208|     6998.0|      67.92|
|    BA| 4296269.199999985|2302.3950696677307|             1866|     6998.0|      67.92|
|    PE|1389734.9500000014| 2285.748273026318|              608|     6998.0|       79.9|
|    CE|1485962.2399999998| 2227.829445277361|              667|     6998.0|      67.92|
|    RJ| 2831397.109999993|2240.0293591772097|             1264|     6998.0|      71.91|
|    DF|1475456.60999

In [29]:
df_agregado2.explain(True)

== Parsed Logical Plan ==
'Aggregate ['estado], ['estado, sum('valor_total) AS faturamento_total#523, avg('valor_total) AS ticket_medio#525, count(1) AS quantidade_vendas#527L, max('valor_total) AS maior_venda#529, min('valor_total) AS menor_venda#531]
+- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_pagamento#412] csv

== Analyzed Logical Plan ==
estado: string, faturamento_total: double, ticket_medio: double, quantidade_vendas: bigint, maior_venda: double, menor_venda: double
Aggregate [estado#411], [estado#411, sum(valor_total#409) AS faturamento_total#523, avg(valor_total#409) AS ticket_medio#525, count(1) AS quantidade_vendas#527L, max(valor_total#409) AS maior_venda#529, min(valor_total#409) AS menor_venda#531]
+- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_

In [30]:
df_agregado3 = (
    df.groupBy("estado", "forma_pagamento")
    .agg(
        F.count("*").alias("quantidade_vendas"),
        F.sum("valor_total").alias("faturamento_total")
    )
        
    )


In [31]:
df_agregado3.show()

+------+---------------+-----------------+------------------+
|estado|forma_pagamento|quantidade_vendas| faturamento_total|
+------+---------------+-----------------+------------------+
|    PR|            Pix|              190|         398349.74|
|    CE|  Cartão Débito|              138|         291080.76|
|    CE|         Boleto|              145|344054.85999999987|
|    DF|            Pix|              202| 475971.4400000001|
|    PE|  Cartão Débito|              129|301570.50000000006|
|    SP|         Boleto|              400| 915608.9700000001|
|    SP|  Cartão Débito|              419| 911452.7400000006|
|    RS| Cartão Crédito|              165| 365503.5999999998|
|    MG|            Pix|              322|         720718.65|
|    BA|            Pix|              575|1292723.2300000014|
|    RJ|  Cartão Débito|              266|         576855.03|
|    GO|         Boleto|              107|213069.55999999994|
|    SP| Cartão Crédito|              593|1349837.4699999997|
|    PE|

In [32]:
df_agregado3.explain(True)

== Parsed Logical Plan ==
'Aggregate ['estado, 'forma_pagamento], ['estado, 'forma_pagamento, count(1) AS quantidade_vendas#599L, sum('valor_total) AS faturamento_total#601]
+- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_pagamento#412] csv

== Analyzed Logical Plan ==
estado: string, forma_pagamento: string, quantidade_vendas: bigint, faturamento_total: double
Aggregate [estado#411, forma_pagamento#412], [estado#411, forma_pagamento#412, count(1) AS quantidade_vendas#599L, sum(valor_total#409) AS faturamento_total#601]
+- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_pagamento#412] csv

== Optimized Logical Plan ==
Aggregate [estado#411, forma_pagamento#412], [estado#411, forma_pagamento#412, count(1) AS quantidade_vendas#599L, sum(valor_total#409) AS faturamento_

In [33]:
df_global = (
    df.agg(
        F.sum("valor_total").alias("faturamento_total"),
        F.avg("valor_total").alias("valor_medio"),
        F.count("*").alias("quantidade_total")
    )
)

In [34]:
df_global.show()

+--------------------+-----------------+----------------+
|   faturamento_total|      valor_medio|quantidade_total|
+--------------------+-----------------+----------------+
|2.2686712640000276E7|2268.671264000028|           10000|
+--------------------+-----------------+----------------+



In [35]:
df_global.explain(True)

== Parsed Logical Plan ==
'Aggregate [sum('valor_total) AS faturamento_total#633, avg('valor_total) AS valor_medio#635, count(1) AS quantidade_total#637L]
+- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_pagamento#412] csv

== Analyzed Logical Plan ==
faturamento_total: double, valor_medio: double, quantidade_total: bigint
Aggregate [sum(valor_total#409) AS faturamento_total#633, avg(valor_total#409) AS valor_medio#635, count(1) AS quantidade_total#637L]
+- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_pagamento#412] csv

== Optimized Logical Plan ==
Aggregate [sum(valor_total#409) AS faturamento_total#633, avg(valor_total#409) AS valor_medio#635, count(1) AS quantidade_total#637L]
+- Project [valor_total#409]
   +- Relation [venda_id#402,data_venda#403,vendedor_id#

In [36]:
df_condicional = (
    df.groupBy("estado")
    .agg(
        F.count("*").alias("quantidade_total"),
        F.sum("valor_total").alias("faturamento_total"),
        F.sum(F.when(F.col("forma_pagamento") == "Pix", 1).otherwise(0)).alias("quantidade_pix"),
        F.sum(F.when(F.col("forma_pagamento") == "Pix", F.col("valor_total")).otherwise(0)).alias("faturamento_pix"),
        F.sum(F.when(F.col("cidade") == "Salvador", F.col("valor_total")).otherwise(0)).alias("faturamento_total_salvador")
        
    )
)

In [37]:
df_condicional.show()

+------+----------------+------------------+--------------+------------------+--------------------------+
|estado|quantidade_total| faturamento_total|quantidade_pix|   faturamento_pix|faturamento_total_salvador|
+------+----------------+------------------+--------------+------------------+--------------------------+
|    GO|             526|1173497.0300000012|           172|412100.91999999987|                       0.0|
|    SP|            1990| 4492084.900000003|           578|1315185.7199999986|                       0.0|
|    RS|             541|1238966.5800000012|           164| 381641.0099999998|                       0.0|
|    MG|            1208|2772609.1599999964|           322|         720718.65|                       0.0|
|    BA|            1866| 4296269.199999985|           575|1292723.2300000014|        1955731.9800000004|
|    PE|             608|1389734.9500000014|           190|434953.82999999996|                       0.0|
|    CE|             667|1485962.2399999998|  

In [38]:
df_condicional.explain(True)

== Parsed Logical Plan ==
'Aggregate ['estado], ['estado, count(1) AS quantidade_total#692L, sum('valor_total) AS faturamento_total#694, sum(CASE WHEN ('forma_pagamento = Pix) THEN 1 ELSE 0 END) AS quantidade_pix#696, sum(CASE WHEN ('forma_pagamento = Pix) THEN 'valor_total ELSE 0 END) AS faturamento_pix#698, sum(CASE WHEN ('cidade = Salvador) THEN 'valor_total ELSE 0 END) AS faturamento_total_salvador#700]
+- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_pagamento#412] csv

== Analyzed Logical Plan ==
estado: string, quantidade_total: bigint, faturamento_total: double, quantidade_pix: bigint, faturamento_pix: double, faturamento_total_salvador: double
Aggregate [estado#411], [estado#411, count(1) AS quantidade_total#692L, sum(valor_total#409) AS faturamento_total#694, sum(CASE WHEN (forma_pagamento#412 = Pix) THEN 1 ELSE 0 END) AS quantidade_pix#696L, sum(CASE WHEN (forma_pa

In [41]:
df_distinct = (
    df.groupBy("estado")
    .agg(
        F.countDistinct("vendedor_id").alias("quantidade_vendedores_distintos")
    )
)

In [42]:
df_distinct.show()

+------+-------------------------------+
|estado|quantidade_vendedores_distintos|
+------+-------------------------------+
|    GO|                             20|
|    SP|                             20|
|    RS|                             20|
|    MG|                             20|
|    BA|                             20|
|    PE|                             20|
|    CE|                             20|
|    RJ|                             20|
|    DF|                             20|
|    PR|                             20|
+------+-------------------------------+



In [43]:
df_distinct.explain(True)

== Parsed Logical Plan ==
'Aggregate ['estado], ['estado, 'count(distinct 'vendedor_id) AS quantidade_vendedores_distintos#793]
+- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_pagamento#412] csv

== Analyzed Logical Plan ==
estado: string, quantidade_vendedores_distintos: bigint
Aggregate [estado#411], [estado#411, count(distinct vendedor_id#404) AS quantidade_vendedores_distintos#793L]
+- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,cidade#410,estado#411,forma_pagamento#412] csv

== Optimized Logical Plan ==
Aggregate [estado#411], [estado#411, count(distinct vendedor_id#404) AS quantidade_vendedores_distintos#793L]
+- Project [vendedor_id#404, estado#411]
   +- Relation [venda_id#402,data_venda#403,vendedor_id#404,produto#405,categoria#406,quantidade#407,valor_unitario#408,valor_total#409,

In [49]:
spark.stop()